# 绘图  

## 简介

本教程描述 **skrf** 的绘图功能。如果你想使用 skrf 的 matplotlib 接口和 skrf 样式，请从以下内容开始

In [ ]:
%matplotlib inline

In [ ]:
from matplotlib import pyplot as plt  # for advanced smith chart only

import skrf as rf

## 绘图方法


绘图函数实现为 `Network` 类的方法。

* `Network.plot_s_re`
* `Network.plot_s_im`
* `Network.plot_s_mag`
* `Network.plot_s_db`
* ...
    
阻抗（`Network.z`）和导纳参数（`Network.y`）也有类似的方法，


* `Network.plot_z_re`
* `Network.plot_z_im`
* ...
* `Network.plot_y_re`
* `Network.plot_y_im`
* ...




## 史密斯圆图


作为第一个示例，加载一个 [Network](../api/network.rst) 并在史密斯圆图上绘制所有四个 s 参数。

In [ ]:
from skrf import Network

ring_slot = Network('data/ring slot.s2p')
ring_slot.plot_s_smith()

scikit-rf 包含一个方便的命令来快速制作更好的图形：

In [ ]:
rf.stylely()  # nicer looking. Can be configured with different styles
ring_slot.plot_s_smith()

In [ ]:
ring_slot.plot_s_smith(draw_labels=True)

另一个常见选项是绘制导纳等高线，而不是阻抗。这通过 `chart_type` 参数控制。

In [ ]:
ring_slot.plot_s_smith(chart_type='y')

### 带标记的高级史密斯圆图

史密斯圆图是一种方便的绘图，显示实部和虚部从零到无穷的复数，并在中心区域放大了实参考值。然而，很难弄清楚哪个点与哪个频率相关，因为没有频率轴。一个常见的解决方法是在表中提供与某些频率标记相关的信息。

In [ ]:
# prepare markers
lines = [
    {'marker_idx': [30, 60, 90], 'color': 'g', 'm': 0, 'n': 0, 'ntw': ring_slot},
    {'marker_idx': [15, 45, 75], 'color': 'r', 'm': 1, 'n': 0, 'ntw': ring_slot},
]

# prepare figure
fig, ax = plt.subplots(1, 1, figsize=(7,8))

# impedance smith chart
rf.plotting.smith(ax = ax, draw_labels = True, ref_imm = 50.0, chart_type = 'z')

# plot data
col_labels = ['Frequency', 'Real Imag']
row_labels = []
row_colors = []
cell_text = []
for l in lines:
    m = l['m']
    n = l['n']
    l['ntw'].plot_s_smith(m=m, n=n, ax = ax, color=l['color'])
    #plot markers
    for i, k in enumerate(l['marker_idx']):
        x = l['ntw'].s.real[k, m, n]
        y = l['ntw'].s.imag[k, m, n]
        z = l['ntw'].z[k, m, n]
        z = f'{z.real:.4f} + {z.imag:.4f}j ohm'
        f = l['ntw'].frequency.f_scaled[k]
        f_unit = l['ntw'].frequency.unit
        row_labels.append(f'M{i + 1}')
        row_colors.append(l['color'])
        ax.scatter(x, y, marker = 'v', s=20, color=l['color'])
        ax.annotate(row_labels[-1], (x, y), xytext=(-7, 7), textcoords='offset points', color=l['color'])
        cell_text.append([f'{f:.3f} {f_unit}', z])
leg1 = ax.legend(loc="upper right", fontsize= 6)

# plot the table
the_table = ax.table(cellText=cell_text,
                      colWidths=[0.4] * 2,
                      rowLabels=row_labels,
                      colLabels=col_labels,
                      rowColours=row_colors,
                      loc='bottom')
the_table.auto_set_font_size(False)
the_table.set_fontsize(6)
#the_table.scale(1.5, 1.5)

### 带背景的高级史密斯圆图

史密斯圆图可以在高分辨率史密斯圆图背景（或你想象的任何其他图像）上方绘制。

In [ ]:
# prepare figure
fig, ax = plt.subplots(1, 1, figsize=(8,8))
background = plt.imread('figures/smithchart.png')

# tweak background position
ax.imshow(background, extent=[-1.185, 1.14, -1.13, 1.155])
rf.plotting.smith(ax = ax, draw_labels = True, ref_imm = 1.0, chart_type = 'z')

ring_slot.plot_s_smith(ax = ax)

    
有关自定义史密斯圆图的更多信息，请参阅 `skrf.plotting.smith()`。

## 复平面


网络参数也可以在复平面上绘制，而不使用史密斯圆图，通过 `Network.plot_s_complex`。

In [ ]:
from matplotlib import pyplot as plt

ring_slot.plot_s_complex()

plt.axis('equal') # otherwise circles won't be circles

## 对数幅值


复网络参数的标量分量也可以相对于频率绘制。要绘制 s 参数的对数幅值与频率的关系，

In [ ]:
ring_slot.plot_s_db()

    
当没有参数传递给绘图方法时，所有参数都会被绘制。可以通过向绘图命令传递索引 `m` 和 `n` 来绘制单个参数（索引从 0 开始）。将环槽的模拟反射系数与测量值进行比较，

In [ ]:
from skrf.data import ring_slot_meas

ring_slot.plot_s_db(m=0,n=0, label='Theory')
ring_slot_meas.plot_s_db(m=0,n=0, label='Measurement')

## 相位


绘制相位，

In [ ]:
ring_slot.plot_s_deg()

    
或展开相位，

In [ ]:
ring_slot.plot_s_deg_unwrap()

弧度（rad）相位也可用

## 群延迟

Network 有一个 `plot()` 方法，它创建参数与频率的矩形图。这可以用来制作不是"预设"的图。例如群延迟

In [ ]:
gd = abs(ring_slot.s21.group_delay) *1e9 # in ns

ring_slot.plot(gd)
plt.ylabel('Group Delay (ns)')
plt.title('Group Delay of Ring Slot S21')

## 阻抗、导纳


阻抗和导纳参数的分量也可以类似地绘制，

In [ ]:
ring_slot.plot_z_im()

In [ ]:
ring_slot.plot_y_im()

## 自定义绘图


图例条目会自动填入 Network 的 `Network.name`。可以通过向绘图方法传递 `label` 参数来覆盖该条目。

In [ ]:
ring_slot.plot_s_db(m=0,n=0, label = 'Simulation')

x 轴上使用的频率单位自动从 Networks 的 `Network.frequency.unit` 属性填充。要更改标签，请更改 frequency 的 `unit`。

In [ ]:
ring_slot.frequency.unit = 'mhz'
ring_slot.plot_s_db(0,0)

传递给绘图方法的其他关键字参数会通过到 matplotlib `matplotlib.pyplot.plot` 函数。

In [ ]:
ring_slot.frequency.unit='ghz'
ring_slot.plot_s_db(m=0,n=0, linewidth = 3, linestyle = '--', label = 'Simulation')
ring_slot_meas.plot_s_db(m=0,n=0, marker = 'o', markevery = 10,label = 'Measured')


绘图的所有组件都可以通过 [matplotlib]( http://matplotlib.sourceforge.net) 函数自定义，并且可以使用上下文管理器使用 `styles`。

In [ ]:
from matplotlib import pyplot as plt
from matplotlib import style

mpl_style = "seaborn-ticks"
mpl_style = mpl_style if mpl_style in style.available else "seaborn-v0_8-ticks"

with style.context(mpl_style):
    ring_slot.plot_s_smith()
    plt.xlabel('Real Part')
    plt.ylabel('Imaginary Part')
    plt.title('Smith Chart With Legend Room')
    plt.axis([-1.1,2.1,-1.1,1.1])
    plt.legend(loc=5)


## 保存绘图

绘图可以使用 matplotlib 提供的 GUI 以各种文件格式保存。然而，skrf 提供了一个便利函数，称为 `skrf.plotting.save_all_figs`，允许将所有打开的图形以多种文件格式保存到磁盘，文件名取自每个图形的标题，

In [ ]:
from skrf.plotting import save_all_figs

save_all_figs('data/', format=['png','eps','pdf'])

## 在绘图后添加标记


一个常见的需求是制作一个彩色绘图，可以在灰度打印中解释。
`skrf.plotting.add_markers_to_lines` 在绘图完成后为图中的每条线添加不同的标记，这通常是你记得要添加它们的时候。

In [ ]:
from skrf import plotting

with plt.style.context('grayscale'):
    ring_slot.plot_s_deg()
    plotting.add_markers_to_lines()
    plt.legend() # have to re-generate legend
